# R6: regenerate the D4 policy-prior observations (CPU runtime)

The D4 observations (`data/r6/<task>-seed<seed>.npz`) were never copied off the
Colab CPU runtime and are lost. D6's encoder agreement gate needs them. This notebook
reruns the recorded CPU run **unchanged**, at the commit all four of its meta files
record (4c3f129), in a separate git worktree. It then checks that the rerun reproduces
the committed `results/r6/returns.csv` and `consistency.csv`.

- **Pass rule:** every value of both CSVs is equal at its full printed precision.
  Byte-level differences are reported separately. `consistency.json` (full float64
  precision), `lens_summary.csv` and the checkpoint hashes are compared for
  information only.
- **MATCH:** the observations go to `MyDrive/layernorm-lens-r6/data/r6/`, and a SHA-256
  manifest and a meta file are written for commit.
- **DIFFERS:** stop and tell the author. Substitute data needs a D7 deviation before
  the gate may use it. `publish` still saves the data and records the verdict, but the
  collector will refuse it.

Nothing here computes a criterion quantity, gate G1 or anything of D6 (b), (c) or (g).
The only numbers produced are the ones the original run produced.

**Runtime > Change runtime type > CPU.** The run takes about 20 minutes plus downloads.
`collect.py consistency` is *expected* to exit with status 2 and print STOP: D3 rejected
identity for cartpole-swingup seed 1 in the original run too.

In [ ]:
# 1. Repository (this session's branch) with its tags. GITHUB_TOKEN Colab secret if private.
import os, subprocess
BRANCH = "claude/new-session-0r0qe0"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = (f"https://{token}@github.com/binoygeorge97/layernorm-lens" if token
       else "https://github.com/binoygeorge97/layernorm-lens")
if not os.path.isdir("/content/layernorm-lens"):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "/content/layernorm-lens"], check=True)
%cd /content/layernorm-lens
!git fetch --tags -q origin
!git log --oneline -1
!for t in prereg-r6 prereg-r6-d1 prereg-r6-d2 prereg-r6-d3; do printf "%-14s " $t; git cat-file -t $t && git rev-parse $t^{commit}; done
!git cat-file -t 4c3f129475adceb0c6e90e56df6696b0565d4986   # the recorded run commit: must print "commit"

In [ ]:
# 2. Google Drive. Outputs go to MyDrive/layernorm-lens-r6/{data,results}/r6/.
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/layernorm-lens-r6"
!ls -la "$DRIVE" "$DRIVE/data/r6" 2>&1 | head -40

In [ ]:
# 3. Environment: a Python 3.13 virtualenv with the pins of the recorded run commit
#    (its requirements.txt and requirements-r6.txt; torch's CPU build from its own index,
#    as that commit's r6_colab.ipynb installed it). The run stage records every
#    installed version next to those in meta_collect.json.
!pip install -q uv
!uv venv -q --python 3.13 /content/d4-venv
!git show 4c3f129:requirements.txt > /content/d4-req.txt && git show 4c3f129:requirements-r6.txt > /content/d4-req-r6.txt
!uv pip install -q --python /content/d4-venv/bin/python torch==2.14.0 --index-url https://download.pytorch.org/whl/cpu
!uv pip install -q --python /content/d4-venv/bin/python -r /content/d4-req.txt -r /content/d4-req-r6.txt
!/content/d4-venv/bin/python -c "import platform, importlib.metadata as m; print('python', platform.python_version()); [print(p, m.version(p)) for p in ('torch','jax','jaxlib','numpy','mujoco','dm_control')]"

In [ ]:
# 4. tdmpc2 at the pinned commit (published returns, results/tdmpc2/<task>.csv).
!test -d checkpoints/tdmpc2_src || git clone -q https://github.com/nicklashansen/tdmpc2 checkpoints/tdmpc2_src
!git -C checkpoints/tdmpc2_src checkout -q e9f59321933cbc8e11a002b842adc7d4ffae8ff1 && git -C checkpoints/tdmpc2_src rev-parse HEAD

In [ ]:
# 5. Run: worktree at 4c3f129, the 15 checkpoints (SHA-256 checked against D6's table, read
#    from the tag), then extract list / extract / lens and collect / consistency with that
#    commit's code and config. Long-running.
!MUJOCO_GL=egl /content/d4-venv/bin/python experiments/r6_tdmpc2/regenerate_d4.py --config experiments/r6_tdmpc2/config.yaml run; echo "exit status: $?"

In [ ]:
# 6. Compare with the committed results. MATCH (exit 0) or DIFFERS (exit 3: stop, tell the author).
!/content/d4-venv/bin/python experiments/r6_tdmpc2/regenerate_d4.py --config experiments/r6_tdmpc2/config.yaml compare; echo "exit status: $?"

In [ ]:
# 7. Publish: observations to Drive (each copy verified), manifest, meta, summaries; prints
#    the git add -f command.
!/content/d4-venv/bin/python experiments/r6_tdmpc2/regenerate_d4.py --config experiments/r6_tdmpc2/config.yaml publish --drive "$DRIVE"; echo "exit status: $?"

In [ ]:
# 8. The files to commit (paths preserved), as a zip. Their Drive copies are in
#    MyDrive/layernorm-lens-r6/results/r6/.
!zip -q -r /content/d4_regeneration.zip results/r6/d4_obs_manifest.csv results/r6/meta_d4_regeneration.json results/r6/d4_regen
!unzip -l /content/d4_regeneration.zip
from google.colab import files
files.download("/content/d4_regeneration.zip")